# Simulating Image Modalities

This tutorial will demonstrate how to simulate a spherical particle with different image modalities and background noise.

In [ ]:
import numpy as np
import deeptrack as dt
from matplotlib import pyplot as plt
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

## Brightfield

In order to simulate a brigthfield modality in a realistic way, we need to resolve several images for different wavelengths and combine the contributions.
We start by preparing a spectrum of wavelengths and a particle scatterer. 

In [ ]:
IMAGE_SIZE = 150

# Generate a spectrum of wavelengths to sample from.
wavelengths = np.linspace(450e-9, 700e-9, 5)

particle = dt.Sphere(
    position=(IMAGE_SIZE//2, IMAGE_SIZE//2, 2) * dt.units.pixel,
    radius= 0.3e-6,
    refractive_index=1.37,
    position_unit="pixel",
    L=10,
)

Image the particle.

In [ ]:
# Sample the wavelengths.
imaged_particle_list = []
for wavelength in wavelengths:
    # Create a brightfield microscope for a given wavelength.
    single_wavelength_optics = dt.Brightfield(
        NA=1.4,
        resolution=1e-6,
        magnification=10,
        wavelength=wavelength,
        padding=(32, 32, 32, 32),
        output_region=(0, 0, IMAGE_SIZE, IMAGE_SIZE),
    )

    # Image the particle.
    imaged_particle = single_wavelength_optics(particle)

    # Add background noise.
    imaged_particle = imaged_particle >> dt.Gaussian(-2, 0.01)

    # Append to list.
    imaged_particle_list.append(imaged_particle)

# Take the average of the images in the list.
averaged_image = (sum(imaged_particle_list) / len(imaged_particle_list)).resolve() 
plt.imshow(averaged_image, cmap="gray")


## Fluorescence
Fluorescence images are easy and straightforward to implement.

In [ ]:
# Create a fluorescence microscope for a given wavelength.
fluorescence_optics = dt.Fluorescence(
    NA=1.4,
    resolution=1e-6,
    magnification=10,
    wavelength=600e-9,
    padding=(32, 32, 32, 32),
    output_region=(0, 0, IMAGE_SIZE, IMAGE_SIZE),
)

# Image the particle.
imaged_particle = fluorescence_optics(particle)

# Add background noise.
imaged_particle_fluorescence = (imaged_particle >> dt.Gaussian(0, 0.0005)).resolve()

plt.imshow(imaged_particle_fluorescence, cmap="gray")

## (Brightfield) Simulating passive Brownian motion
We can combine the optical tools together with equations of motion to simulate a realistic movie of a particle diffusing with Brownian motion.

In [ ]:
# Make image bigger.
IMAGE_SIZE = 150

simulated_movie = []
movie_length = 50
dx, dy, dz = 0, 0, 0
for t in range(movie_length):
    
    imaged_particle_list = []

    # Create a particle with a given position.
    particle = dt.Sphere(
        position=(
            IMAGE_SIZE//2 + dx,
            IMAGE_SIZE//2 + dy,
            2 + dz
        ) * dt.units.pixel,
        radius= 0.3e-6,
        refractive_index=1.37,
        position_unit="pixel",
        L=10,
    )      

    # Sample wavelengths to obtain a good brightfield image.
    for wavelength in wavelengths:
        single_wavelength_optics = dt.Brightfield(
            NA=1.4,
            resolution=1e-7,
            magnification=2,
            wavelength=wavelength,
            padding=(32, 32, 32, 32),
            output_region=(0, 0, IMAGE_SIZE, IMAGE_SIZE),
        )

        # Image particle and add background noise.
        imaged_particle_list.append(
            single_wavelength_optics(particle) >> dt.Gaussian(-2, 0.01)
        )

    # Average the images.
    image = (sum(imaged_particle_list) / len(imaged_particle_list)).resolve() 

    # Add to list of frames.
    simulated_movie.append(image)

    # Update the displacements.
    dx += np.random.standard_normal()*0.6
    dy += np.random.standard_normal()*0.6
    dz += np.random.standard_normal()*0.1

plt.imshow(simulated_movie[0],cmap="gray")


## Make an animation from the simulation.
Using `FuncAnimation` and a `HTML` player, we will combine the simulated frames into a movie.

In [ ]:
# Create figure to display frames.
fig, ax = plt.subplots(figsize=(8, 8))

# Function to update the figure for a given frame.
def update(frame):

    display_frame = simulated_movie[frame]
    ax.clear()
    ax.set_axis_off()
    ax.imshow(display_frame, cmap="gray")
    ax.set_xlim(0, display_frame.shape[1])
    ax.set_ylim(display_frame.shape[0], 0)
    ax.set_title("Passive Brownian Motion with Brightfield")

    return ax

# Make animation.
animation = FuncAnimation(
    fig,
    update,
    frames=len(simulated_movie)
)

# Animation player as a HTML object.
player = HTML(animation.to_jshtml(fps=10)); plt.close()

# Call player to view the movie. 
player

## (Fluorescence) Simulating passive Brownian motion
Using the same method as above, we will now simulate passive Brownian motion with a fluorescence image modality.

In [ ]:
# Define optics.
fluorescence_optics = dt.Fluorescence(
    NA=1.4,
    resolution=1e-7,
    magnification=2,
    wavelength=600e-9,
    padding=(32, 32, 32, 32),
    output_region=(0, 0, IMAGE_SIZE, IMAGE_SIZE),
)

simulated_movie = []
movie_length = 100
dx, dy, dz = 0, 0, 0
for t in range(movie_length):
    
    imaged_particle_list = []

    # Create a particle with a given position.
    particle = dt.Sphere(
        position=(IMAGE_SIZE//2 + dx,
                  IMAGE_SIZE//2 + dy,
                  2 + dz) * dt.units.pixel,
        radius= 0.3e-6,
        refractive_index=1.37,
        position_unit="pixel",
        L=10,
    )      

    # Sample wavelengths to obtain a good brightfield image.
    imaged_particle = (
        fluorescence_optics(particle)
        >> dt.Gaussian(-2, 0.0005)
    ).resolve()

    # Add to list of frames.
    simulated_movie.append(imaged_particle)

    # Update the displacements.
    dx += np.random.standard_normal()*0.6
    dy += np.random.standard_normal()*0.6
    dz += np.random.standard_normal()*0.3

plt.imshow(simulated_movie[0],cmap="gray")


## Make an animation from the simulation.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

def update(frame):
    display_frame = simulated_movie[frame]
    ax.clear()
    ax.set_axis_off()
    ax.imshow(display_frame, cmap="gray")
    ax.set_xlim(0, display_frame.shape[1])
    ax.set_ylim(display_frame.shape[0], 0)
    ax.set_title("Passive Brownian Motion with Fluorescence")

    return ax

# Make and display animation.
animation = FuncAnimation(
    fig,
    update,
    frames=len(simulated_movie)
)
video = HTML(animation.to_jshtml(fps=10)); plt.close()
video